<a href="https://colab.research.google.com/github/tjdux/Introduction-to-Machine-Learning-with-Python/blob/main/06_6_%EB%AA%A8%EB%8D%B8_%EC%84%A0%ED%83%9D%EC%9D%84_%EC%9C%84%ED%95%9C_%EA%B7%B8%EB%A6%AC%EB%93%9C_%EC%84%9C%EC%B9%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- 파이프라인을 구상하는 단계도 탐색 대상으로 삼을 수 있음 (e.g. StandardScaler와 MinMaxScaler 중 어느 것을 사용할지)
- 이제부터 `RandomForestClassifier`와 `SVC`를 비교한다고 가정 (`SVC`는 데이터 스케일을 조정해야 하므로 `StandardScaler`를 사용할지 또는 전처리를 하지 않을지도 판단)

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

pipe = Pipeline([("preprocessing", StandardScaler()), ("classifier", SVC())])

In [3]:
from sklearn.ensemble import RandomForestClassifier

param_grid = [
    {"classifier": [SVC()], "preprocessing": [StandardScaler()],
     "classifier__gamma": [0.001, 0.01, 0.1, 1, 10, 100],
     "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100]},
    {"classifier": [RandomForestClassifier(n_estimators=100)],
     "preprocessing": [None], "classifier__max_features": [1, 2, 3]}
]

In [7]:
# 데이터 로드
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0
)

In [9]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)

print(f"최적의 매개변수:\n{grid.best_params_}\n")
print(f"최상의 교차 검증 점수: {round(grid.best_score_, 2)}")
print(f"테스트 세트 점수: {round(grid.score(X_test, y_test), 2)}")

최적의 매개변수:
{'classifier': SVC(), 'classifier__C': 10, 'classifier__gamma': 0.01, 'preprocessing': StandardScaler()}

최상의 교차 검증 점수: 0.99
테스트 세트 점수: 0.98


### 01 중복 계산 피하기
- `memory` 매개변수: 계산 결과를 캐싱

In [10]:
pipe = Pipeline([('preprocessing', StandardScaler()), ("classifier", SVC())],
                memory="cache_folder")

- 단점
  - 비교적 오랜 시간이 걸리는 변환이어야 `memory` 매개변수를 사용하여 속도를 높이는 효과를 낼 수 있음
  - `n_jobs` 매개변수가 캐싱을 방해
- `dask-ml` 라이브러리에서 제공하는 `GridSearchCV`를 사용하면 이런 단점들을 피할 수 있음